In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import plot_to_tensorboard, count_parameters, MLPClassifier
import pandas as pd
import hashlib
from pathlib import Path
import mlflow
import mlflow.pytorch
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils
from sklearn.model_selection import train_test_split
from collections import Counter
import random


c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
mlflow.set_experiment("Clasificador_Imagenes_HP_Search")

<Experiment: artifact_location='file:///c:/ITBA/REDES%20NEURONALES/Tp1-Redes-Neuronales/mlruns/658278433750724451', creation_time=1779393889228, experiment_id='658278433750724451', last_update_time=1779393889228, lifecycle_stage='active', name='Clasificador_Imagenes_HP_Search', tags={}>

In [3]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    labels_presentes = sorted(list(set(list(all_labels) + list(all_preds))))
    classes_presentes = [classes[i] for i in labels_presentes]
    
    matriz_conf = confusion_matrix(all_labels, all_preds, labels=labels_presentes)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=matriz_conf, display_labels=classes_presentes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)
    plt.close(fig_cm)

    cls_report = classification_report(all_labels, all_preds, labels=labels_presentes, target_names=classes_presentes)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    report_path = f"classification_report_{prefix}_epoch_{step}.txt"
    with open(report_path, "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(report_path)
    os.remove(report_path)

In [4]:
def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    model.eval()  # Modo evaluación activo
    model.to(device) # Asegura el modelo en GPU/CPU
    
    log_classification_report(model, loader, writer, device, classes, step=epoch, prefix=prefix)

    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [5]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

        # Extrae las clases según los nombres de las carpetas de las rutas filtradas
        self.classes = sorted(list(set([Path(p).parent.name for p in self.image_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label  # <- Imagen en 3D (C, H, W). El modelo se encarga de aplanar

In [6]:
# 1. JUNTAR TODO
data_dir_total = Path("data/Split_smol")
all_paths = [p for p in data_dir_total.glob("**/*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

def get_class(x):
    return x.parent.name

all_files_data = []
for p in all_paths:
    try:
        with Image.open(p) as img:
            all_files_data.append((p, get_class(p)))
    except Exception:
        pass

df_total = pd.DataFrame(all_files_data, columns=["path", "class"]).sort_values(by="path").reset_index(drop=True)

# 2. SACAR FOTOS PROBLEMÁTICAS Y DUPLICADOS
fotos_a_eliminar = {"aug_0_F2.large.jpg"}  # foto negra

hashes_vistos = set()
indices_a_mantener = []

for idx, row in df_total.iterrows():
    p = row["path"]

    # Filtro por nombre
    if p.name in fotos_a_eliminar:
        continue

    # Filtro por hash
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash not in hashes_vistos:
        hashes_vistos.add(file_hash)
        indices_a_mantener.append(idx)

df_limpio = df_total.loc[indices_a_mantener].reset_index(drop=True)
print(f"Total imágenes limpias: {len(df_limpio)}")

# 3. SPLIT 60 / 20 / 20
all_image_paths = [str(p) for p in df_limpio["path"].tolist()]
all_labels = df_limpio["class"].tolist()

SEED = 42

train_val_paths, test_image_paths, train_val_labels, _ = train_test_split(
    all_image_paths, all_labels, test_size=0.20, random_state=SEED, stratify=all_labels
)

train_image_paths, val_image_paths, _, _ = train_test_split(
    train_val_paths, train_val_labels, test_size=0.25, random_state=SEED, stratify=train_val_labels
)

print(f"Train: {len(train_image_paths)} | Val: {len(val_image_paths)} | Test: {len(test_image_paths)}")

Total imágenes limpias: 842
Train: 504 | Val: 169 | Test: 169


In [7]:
counts = Counter([Path(p).parent.name for p in train_image_paths])
max_count = max(counts.values())

for cls, count in counts.items():
    faltantes = max_count - count
    if faltantes > 0:
        paths_cls = [p for p in train_image_paths if Path(p).parent.name == cls]
        train_image_paths.extend(random.choices(paths_cls, k=faltantes))

print(f"Train después del oversampling: {len(train_image_paths)}")

counts_post = Counter([Path(p).parent.name for p in train_image_paths])
for cls, count in sorted(counts_post.items()):
    print(f"  {cls}: {count}")

Train después del oversampling: 540
  Actinic keratosis: 60
  Atopic Dermatitis: 60
  Benign keratosis: 60
  Dermatofibroma: 60
  Melanocytic nevus: 60
  Melanoma: 60
  Squamous cell carcinoma: 60
  Tinea Ringworm Candidiasis: 60
  Vascular lesion: 60


In [8]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin_hp"
writer  = SummaryWriter(log_dir=log_dir)

In [9]:
hparams_space = {
    "input_size":  [32, 64],       
    "batch_size":  [16, 64],       
    "lr":          [1e-3, 1e-4],  
    "epochs":      200,                 
    "optimizer":   ["Adam", "SGD"],     
    "momentum":    [0.9, 0.99],         
    "HFlip":       [0.0, 0.5],          
    "VFlip":       [0.0, 0.5],          
    "RBContrast":  [0.0, 0.5],          
    "CLAHE":       [0.0, 0.3],          # <--- ¡NUEVO! Probabilidad de CLAHE (Contraste Médico)
    "CDropout":    [0.0, 0.3],          # <--- ¡NUEVO! Probabilidad de CoarseDropout (Parches Negros)
    "dropout":     [0.0, 0.1, 0.2, 0.3],
    "es_patience": 5,                   
    "init_weights": ["Default", "Kaiming"] 
}

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelnbr = 0
for input_size in hparams_space["input_size"]:
    for batch_size in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer_name in hparams_space["optimizer"]:
                for momentum_val in hparams_space["momentum"]:
                    for hflip_p in hparams_space["HFlip"]:
                        for vflip_p in hparams_space["VFlip"]:
                            for rbc_p in hparams_space["RBContrast"]:
                                for clahe_p in hparams_space["CLAHE"]:       # Nuevo bucle CLAHE
                                    for cdropout_p in hparams_space["CDropout"]: # Nuevo bucle CoarseDropout
                                        for dropout in hparams_space["dropout"]:
                                            for init_method in hparams_space["init_weights"]:

                                                # Evitamos redundancia: Adam no usa el momentum manual de SGD
                                                if optimizer_name == "Adam" and momentum_val == 0.99:
                                                    continue
                                                
                                                # FILTRO DEL AZAR: Ajustalo a 0.05 (5%) o 0.10 (10%) según tu prisa
                                                if np.random.rand() >= 0.05: 
                                                    continue  

                                                modelnbr += 1
                                                print(f"\n>>> Modelo N° {modelnbr} | Size: {input_size} | Opt: {optimizer_name} | Init: {init_method} <<<")
                                                print(f"    Probabilidades -> CLAHE: {clahe_p} | CoarseDrop: {cdropout_p} | RBContrast: {rbc_p}")

                                                # Diccionario para MLflow con los nuevos parámetros
                                                hparams = {
                                                    "model":         "MLPClassifier",
                                                    "input_size":    input_size * input_size * 3,
                                                    "batch_size":    batch_size,
                                                    "lr":            lr,
                                                    "epochs":        hparams_space["epochs"],
                                                    "optimizer":     optimizer_name,
                                                    "momentum":      momentum_val if optimizer_name == "SGD" else "N/A",
                                                    "loss_fn":       "CrossEntropyLoss",
                                                    "data_dir": data_dir_total,
                                                    "n_train": len(train_image_paths),
                                                    "n_val": len(val_image_paths),
                                                    "es_patience":   hparams_space["es_patience"],
                                                    "dropout":       dropout,
                                                    "p_HFlip":       hflip_p,
                                                    "p_VFlip":       vflip_p,
                                                    "p_RBContrast":  rbc_p,
                                                    "p_CLAHE":       clahe_p,
                                                    "p_CoarseDrop":  cdropout_p,
                                                    "init_weights":  init_method
                                                }

                                                # CONSTRUCCIÓN DINÁMICA DE ALBUMENTATIONS CON CLAHE Y CDROPOUT
                                                train_transform = A.Compose([
                                                    A.Resize(input_size, input_size),
                                                    A.HorizontalFlip(p=hflip_p),
                                                    A.VerticalFlip(p=vflip_p),
                                                    A.RandomBrightnessContrast(p=rbc_p),
                                                    A.CLAHE(p=clahe_p),                                                          
                                                    A.CoarseDropout(max_holes=6, max_height=1, max_width=20, p=cdropout_p, fill_value=0),  
                                                    A.Normalize(),
                                                    ToTensorV2()
                                                ])
                                                
                                                val_transform = A.Compose([
                                                    A.Resize(input_size, input_size),
                                                    A.Normalize(),
                                                    ToTensorV2()
                                                ])

                                                # Datasets y loaders
                                                train_dataset = CustomImageDataset(train_image_paths, transform=train_transform)
                                                val_dataset   = CustomImageDataset(val_image_paths,   transform=val_transform)

                                                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                                                val_loader   = DataLoader(val_dataset,   batch_size=batch_size)

                                                num_classes = len(train_dataset.classes)
                                                
                                                model = MLPClassifier(
                                                    num_classes=num_classes,
                                                    input_size=input_size * input_size * 3
                                                ).to(device)
                                                
                                                # Modificación de dropouts secuenciales internos
                                                if len(model.model) > 4 and isinstance(model.model[4], nn.Dropout):
                                                    model.model[4].p = dropout
                                                if len(model.model) > 9 and isinstance(model.model[9], nn.Dropout):
                                                    model.model[9].p = dropout

                                                # Inicialización de pesos He
                                                if init_method == "Kaiming":
                                                    for layer in model.modules():
                                                        if isinstance(layer, nn.Linear):
                                                            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                                                            if layer.bias is not None:
                                                                nn.init.constant_(layer.bias, 0.0)

                                                criterion = nn.CrossEntropyLoss()
                                                
                                                if optimizer_name == "Adam":
                                                    optimizer = optim.Adam(model.parameters(), lr=lr)
                                                else:
                                                    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum_val)

                                                hparams["count_params"] = count_parameters(model)

                                                # Tracking MLflow
                                                with mlflow.start_run():
                                                    mlflow.log_params(hparams)

                                                    best_val_acc   = 0
                                                    best_val_loss  = float('inf')
                                                    best_train_acc = 0
                                                    best_train_loss = 0
                                                    best_epoch     = 0

                                                    for epoch in range(hparams["epochs"]):
                                                        model.train()
                                                        running_loss = 0.0
                                                        correct, total = 0, 0

                                                        for images, labels in train_loader:
                                                            images, labels = images.to(device), labels.to(device)

                                                            optimizer.zero_grad()
                                                            outputs = model(images)
                                                            loss = criterion(outputs, labels)
                                                            loss.backward()
                                                            optimizer.step()

                                                            running_loss += loss.item()
                                                            _, preds = torch.max(outputs, 1)
                                                            correct += (preds == labels).sum().item()
                                                            total   += labels.size(0)

                                                        train_loss = running_loss / len(train_loader)
                                                        train_acc  = 100.0 * correct / total
                                                        
                                                        val_loss, val_acc = evaluate(
                                                            model, val_loader, writer, device,
                                                            train_dataset.classes, criterion,
                                                            epoch=epoch, prefix="val"
                                                        )

                                                        writer.add_scalar("train/loss",     train_loss, epoch)
                                                        writer.add_scalar("train/accuracy", train_acc,  epoch)

                                                        mlflow.log_metrics({
                                                            "train_loss":     train_loss,
                                                            "train_accuracy": train_acc,
                                                            "val_loss":       val_loss,
                                                            "val_accuracy":   val_acc
                                                        }, step=epoch)

                                                        if val_acc > best_val_acc:
                                                            best_val_acc    = val_acc
                                                            best_val_loss   = val_loss
                                                            best_train_acc  = train_acc
                                                            best_train_loss = train_loss
                                                            best_epoch      = epoch
                                                            
                                                            torch.save(model.state_dict(), "best_mlp_model.pth")
                                                            mlflow.log_artifact("best_mlp_model.pth")
                                                            mlflow.pytorch.log_model(model, artifact_path="pytorch_model")

                                                        elif epoch > best_epoch + hparams["es_patience"]:
                                                            print(f" -> Early Stopping en época {epoch+1}. Volviendo a epoch {best_epoch+1}.")
                                                            break

                                                    mlflow.log_metrics({
                                                        "best_train_loss": best_train_loss,
                                                        "best_train_acc":  best_train_acc,
                                                        "best_val_loss":   best_val_loss,
                                                        "best_val_acc":    best_val_acc,
                                                        "best_epoch":      best_epoch
                                                    }, step=epoch + 1)

print(f"\n Búsqueda terminada con éxito. Se procesaron {modelnbr} configuraciones.")


>>> Modelo N° 1 | Size: 32 | Opt: Adam | Init: Default <<<
    Probabilidades -> CLAHE: 0.0 | CoarseDrop: 0.0 | RBContrast: 0.0


C:\Users\Sofia\AppData\Local\Temp\ipykernel_1212\2298217467.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=1, max_width=20, p=cdropout_p, fill_value=0),
2026/05/29 13:47:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 13:47:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])

 -> Early Stopping en época 16. Volviendo a epoch 10.

>>> Modelo N° 2 | Size: 32 | Opt: Adam | Init: Kaiming <<<
    Probabilidades -> CLAHE: 0.0 | CoarseDrop: 0.3 | RBContrast: 0.0


C:\Users\Sofia\AppData\Local\Temp\ipykernel_1212\2298217467.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=1, max_width=20, p=cdropout_p, fill_value=0),
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-p

 -> Early Stopping en época 21. Volviendo a epoch 15.

>>> Modelo N° 3 | Size: 32 | Opt: Adam | Init: Default <<<
    Probabilidades -> CLAHE: 0.3 | CoarseDrop: 0.0 | RBContrast: 0.0


...

 -> Early Stopping en época 22. Volviendo a epoch 16.

>>> Modelo N° 8 | Size: 32 | Opt: Adam | Init: Kaiming <<<
    Probabilidades -> CLAHE: 0.0 | CoarseDrop: 0.0 | RBContrast: 0.0


C:\Users\Sofia\AppData\Local\Temp\ipykernel_1212\2298217467.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=1, max_width=20, p=cdropout_p, fill_value=0),
2026/05/29 14:29:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:30:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:31:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:31:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when 

 -> Early Stopping en época 11. Volviendo a epoch 5.

>>> Modelo N° 9 | Size: 32 | Opt: SGD | Init: Default <<<
    Probabilidades -> CLAHE: 0.0 | CoarseDrop: 0.3 | RBContrast: 0.0


C:\Users\Sofia\AppData\Local\Temp\ipykernel_1212\2298217467.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=1, max_width=20, p=cdropout_p, fill_value=0),
2026/05/29 14:34:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:35:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:35:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/29 14:36:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when 

RuntimeError: File best_mlp_model.pth cannot be opened.